In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="03-google-street-view",
    notebook_name="01_street_view_system_design.ipynb"
)

# Google Street View Blurring System -- Full ML System Design

## The Big Idea (For a 12-Year-Old)

Imagine Google has special cars that drive down every street in the world, snapping pictures. These pictures go on Google Maps so you can peek at any street before visiting. But here is the problem: the cameras capture people's **faces** and car **license plates** -- that is private information! Nobody wants their face plastered all over the internet without permission.

So Google needs a super-smart computer program that can:
1. Look at **millions** of photos
2. Find every face and every license plate
3. Blur them out -- like putting a fuzzy sticker over them

And it needs to do this **before** anyone sees the photos. Think of it like a privacy guard that checks every photo before it goes live.

---

## Staff-Level Technical Summary

Google Street View's blurring system is an **offline batch-processing object detection pipeline** that identifies and obscures personally identifiable information (PII) -- specifically human faces and license plates -- in street-level panoramic imagery. The system prioritizes **recall** (catching every face/plate) over precision, because a missed face is a **privacy violation**, while a false positive blur is merely a cosmetic issue. The system operates offline, meaning latency is not a primary concern.

This notebook walks through the complete 7-step ML system design framework:
1. Clarify requirements
2. Frame as ML task
3. Data preparation
4. Model development
5. Evaluation
6. Serving
7. Discussion topics

## Step 1: Clarify Requirements

### The Interview Starts Here

In a real interview, you **never** jump straight into a solution. You ask questions first. Think of it like a detective gathering clues before solving the case.

In [ ]:
# Let's organize the clarifying requirements as structured data
# This is how you'd think about it systematically in an interview

requirements = {
    "business_objective": "Protect user privacy by blurring PII in street view images",
    "pii_types": ["human faces", "license plates"],
    "processing_mode": "Offline batch processing (NOT real-time)",
    "latency_constraint": "None -- existing images shown while new ones process in background",
    "training_data": "1 million images with bounding box annotations",
    "feedback_mechanism": "Users can report incorrectly blurred (or missed) images",
    "bias_concerns": "Face detection must work equally well across races, ages, genders",
    "key_tradeoff": "Recall >> Precision (missing a face = privacy violation, false blur = cosmetic issue)"
}

print("=== Google Street View Blurring System Requirements ===")
for key, value in requirements.items():
    print(f"\n  {key}:")
    if isinstance(value, list):
        for item in value:
            print(f"    - {item}")
    else:
        print(f"    {value}")

print("\n" + "="*60)
print("KEY INSIGHT: Because this is OFFLINE processing, we can use")
print("the most accurate (but slower) models without worrying about")
print("latency. This is a huge advantage over real-time systems.")

### Why Recall Matters More Than Precision Here

Imagine you are a photo reviewer. You have two types of mistakes:

| Mistake Type | What Happens | Example | Severity |
|---|---|---|---|
| **False Negative** (missed face) | A real face is NOT blurred | Someone's face is visible on Google Maps | **CRITICAL** -- privacy violation, potential lawsuit |
| **False Positive** (blurred non-face) | A non-face IS blurred | A round sign gets blurred unnecessarily | **Minor** -- cosmetic, nobody is harmed |

This asymmetry means: **when in doubt, blur it!** We optimize for high recall (catching everything) and accept some extra false positives.

## Step 2: Frame as an ML Task

### Simple Explanation

Think of it like a game of "I Spy." You show the computer a photo and say: "Find all the faces and license plates." The computer draws rectangles around each one it finds (called **bounding boxes**). Then a blurring tool smudges everything inside those rectangles.

### Technical Framing

- **Input**: A street-level image (containing zero or more faces/license plates)
- **Output**: A list of bounding boxes, each with:
  - Position: `(x, y)` coordinates of the top-left corner
  - Size: `width` and `height`
  - Class: `"human_face"` or `"license_plate"`
  - Confidence score (0 to 1)

### ML Category: Object Detection

Object detection = **Localization** (where is it?) + **Classification** (what is it?)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# === Visualize what object detection output looks like ===

np.random.seed(42)

# Create a synthetic "street view" image (just colored noise to represent an image)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Generate a synthetic image (gradient + noise to simulate a scene)
img = np.zeros((300, 500, 3), dtype=np.uint8)
# Sky (top half - light blue)
img[:150, :] = [135, 206, 235]
# Ground (bottom half - gray)
img[150:, :] = [128, 128, 128]
# Building-like block
img[50:250, 50:150] = [180, 160, 140]
img[50:250, 200:350] = [160, 140, 130]
# "Faces" (small flesh-colored rectangles)
img[100:140, 80:110] = [220, 180, 150]  # face 1
img[120:155, 250:275] = [200, 165, 135]  # face 2
img[180:195, 320:360] = [200, 200, 200]  # license plate area
# Add some noise
noise = np.random.randint(-10, 10, img.shape, dtype=np.int16)
img = np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)

# Left: original image with no detection
axes[0].imshow(img)
axes[0].set_title("Original Street View Image", fontsize=13, fontweight='bold')
axes[0].axis('off')

# Right: image with bounding boxes drawn
axes[1].imshow(img)
axes[1].set_title("After Object Detection", fontsize=13, fontweight='bold')
axes[1].axis('off')

# Draw bounding boxes
detections = [
    {"bbox": [70, 90, 50, 60], "class": "face", "conf": 0.95, "color": "red"},
    {"bbox": [240, 110, 45, 55], "class": "face", "conf": 0.88, "color": "red"},
    {"bbox": [310, 172, 55, 28], "class": "plate", "conf": 0.92, "color": "blue"},
]

for det in detections:
    x, y, w, h = det["bbox"]
    rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor=det["color"], facecolor='none')
    axes[1].add_patch(rect)
    axes[1].text(x, y - 5, f"{det['class']} ({det['conf']:.2f})",
                 color=det["color"], fontsize=9, fontweight='bold',
                 bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.show()

print("\nEach detection has:")
print("  - Bounding box: (x, y, width, height) -- WHERE the object is")
print("  - Class: face or license_plate -- WHAT the object is")
print("  - Confidence: 0 to 1 -- HOW SURE the model is")

## Step 3: Architecture Families

There are three main families of object detection models. Think of them as different strategies for finding things in a photo:

### Two-Stage Detectors (Careful and Accurate)

**12-year-old version**: Imagine looking for your friend in a crowded stadium. First, you scan the whole crowd and circle sections that MIGHT have your friend (Stage 1). Then, you look closely at each circled section to check if your friend is actually there (Stage 2).

**Examples**: R-CNN, Fast R-CNN, **Faster R-CNN**

### One-Stage Detectors (Fast and Good Enough)

**12-year-old version**: Instead of scanning then checking, you just look at the whole picture all at once and instantly say "face here, plate there."

**Examples**: **YOLO** (You Only Look Once), **SSD** (Single Shot MultiBox Detector)

### Transformer-Based (Newer Approach)

**Examples**: **DETR** (DEtection TRansformer) -- treats detection as a set prediction problem.

In [ ]:
# === Compare the three architecture families ===

architectures = {
    "Two-Stage (Faster R-CNN)": {
        "how_it_works": "Stage 1: RPN proposes regions -> Stage 2: Classify each region",
        "speed": "Slow (5-7 FPS)",
        "accuracy": "Highest (mAP ~42 on COCO)",
        "best_for": "Accuracy-critical offline tasks",
        "analogy": "Detective who searches room by room carefully"
    },
    "One-Stage (YOLO/SSD)": {
        "how_it_works": "Single pass: predict boxes + classes simultaneously on a grid",
        "speed": "Fast (30-60+ FPS)",
        "accuracy": "Good (mAP ~37-40 on COCO)",
        "best_for": "Real-time applications",
        "analogy": "Quick glance that spots everything at once"
    },
    "Transformer (DETR)": {
        "how_it_works": "Encodes image with CNN, decodes objects with Transformer attention",
        "speed": "Moderate",
        "accuracy": "Competitive (mAP ~42+ on COCO)",
        "best_for": "When you want to avoid hand-designed components (no NMS, no anchors)",
        "analogy": "AI that learns the whole game from scratch without rules"
    }
}

print("=== Object Detection Architecture Families ===")
for name, details in architectures.items():
    print(f"\n{'='*55}")
    print(f"  {name}")
    print(f"{'='*55}")
    for key, value in details.items():
        print(f"  {key}: {value}")

print("\n" + "="*55)
print("\nOUR CHOICE: Two-Stage (Faster R-CNN)")
print("\nWhy?")
print("  1. Privacy demands MAXIMUM accuracy (recall is critical)")
print("  2. Processing is OFFLINE -- no real-time latency constraint")
print("  3. Dataset is 1M images -- manageable training cost")
print("  4. If we scale to 100M+ images, we can switch to YOLO later")

## Step 4: Data Pipeline

### Data Sources

We have two types of data:

1. **Annotated dataset**: 1 million images with bounding boxes labeled by humans
2. **Raw street view images**: Unlabeled production images from the camera cars

### Data Augmentation

**12-year-old version**: Imagine you have one photo of a cat. If you flip it, rotate it slightly, make it brighter, and make it darker, you now have FIVE photos of a cat. The computer learns to recognize cats in many different conditions, making it much smarter.

**Critical rule**: When you augment an image, you MUST also transform the bounding boxes. If you flip the image, the bounding box coordinates flip too!

In [ ]:
# === Demonstrate data augmentation with bounding box transformation ===

def create_sample_image_with_box():
    """Create a synthetic image with a bounding box (simulating a face)."""
    img = np.zeros((200, 300, 3), dtype=np.uint8)
    # Background gradient
    for i in range(200):
        img[i, :] = [100 + i//4, 150 - i//5, 180 - i//4]
    # "Face" region
    img[60:120, 100:160] = [220, 180, 150]
    # Eyes (dark dots)
    img[80:85, 115:120] = [50, 50, 50]
    img[80:85, 140:145] = [50, 50, 50]
    # Mouth
    img[100:105, 120:140] = [180, 100, 100]
    bbox = [100, 60, 60, 60]  # x, y, w, h
    return img, bbox


def augment_horizontal_flip(img, bbox):
    """Flip image and transform bounding box."""
    flipped = np.fliplr(img).copy()
    x, y, w, h = bbox
    new_x = img.shape[1] - x - w  # Mirror x coordinate
    return flipped, [new_x, y, w, h]


def augment_brightness(img, bbox, factor=1.3):
    """Change brightness (bounding box stays the same)."""
    bright = np.clip(img.astype(float) * factor, 0, 255).astype(np.uint8)
    return bright, bbox  # Bbox unchanged!


def augment_add_noise(img, bbox, noise_level=20):
    """Add random noise to simulate different camera conditions."""
    noise = np.random.randint(-noise_level, noise_level, img.shape, dtype=np.int16)
    noisy = np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    return noisy, bbox  # Bbox unchanged!


# Create original and augmented versions
img, bbox = create_sample_image_with_box()
flip_img, flip_bbox = augment_horizontal_flip(img, bbox)
bright_img, bright_bbox = augment_brightness(img, bbox)
noisy_img, noisy_bbox = augment_add_noise(img, bbox)

# Plot all versions
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
titles = ["Original", "Horizontal Flip", "Brightness +30%", "Random Noise"]
images = [img, flip_img, bright_img, noisy_img]
bboxes = [bbox, flip_bbox, bright_bbox, noisy_bbox]

for ax, title, im, bb in zip(axes, titles, images, bboxes):
    ax.imshow(im)
    x, y, w, h = bb
    rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor='red', facecolor='none')
    ax.add_patch(rect)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.axis('off')

plt.suptitle("Data Augmentation with Bounding Box Transformation", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Original bbox:     ", bbox)
print("Flipped bbox:      ", flip_bbox, " <-- x coordinate changed!")
print("Bright bbox:       ", bright_bbox, " <-- unchanged (only pixels changed)")
print("Noisy bbox:        ", noisy_bbox, " <-- unchanged (only pixels changed)")
print("\nWith 4 augmentations, 1M images become ~5M images.")
print("With 10 augmentations, 1M becomes ~10M. This helps the model generalize.")

## Step 5: Model Architecture -- Faster R-CNN Deep Dive

### The Three Components

```
Input Image
    |
    v
+-------------------+
| 1. BACKBONE CNN   |  (e.g., ResNet-50)
| Extracts features |  Input: raw pixels
| from the image    |  Output: feature map
+-------------------+
    |
    v
+-------------------+
| 2. REGION         |  Slides over feature map
| PROPOSAL NETWORK  |  Proposes ~2000 candidate regions
| (RPN)             |  "These areas MIGHT have objects"
+-------------------+
    |
    v
+-------------------+
| 3. DETECTION HEAD |  Classifies each proposed region
| (Classifier +     |  Refines bounding box coordinates
|  Box Regressor)   |  Output: class + precise bbox
+-------------------+
```

### Loss Function

Object detection uses a **combined loss** because it does two things at once:

- **Classification loss** (cross-entropy): "Is this a face, plate, or background?"
- **Regression loss** (MSE/Smooth L1): "How close is my bounding box to the ground truth?"

```
Total Loss = Classification Loss + lambda * Regression Loss
```

Where `lambda` balances the two losses.

In [ ]:
# === Simulate the Faster R-CNN pipeline stages ===

def simulate_backbone(image_size=(800, 800), stride=16):
    """
    Simulate what a backbone CNN does.
    
    12-year-old version:
    The backbone is like reading a book and writing a summary.
    It takes a big image and compresses it into a smaller 'feature map'
    that captures the important patterns (edges, shapes, textures).
    
    Technical detail:
    A ResNet-50 backbone with stride 16 converts an 800x800 image
    into a 50x50 feature map with 256 channels.
    """
    h, w = image_size
    feature_h = h // stride
    feature_w = w // stride
    channels = 256  # Typical for ResNet
    
    # Simulate feature map (random activations)
    feature_map = np.random.randn(channels, feature_h, feature_w).astype(np.float32)
    
    return feature_map


def simulate_rpn(feature_map, num_anchors=9, nms_threshold=0.7, top_k=300):
    """
    Simulate Region Proposal Network.
    
    12-year-old version:
    The RPN slides a magnifying glass across the feature map.
    At each spot, it checks 9 different rectangle shapes and says
    'Is there an object here? Maybe? Probably not?'
    It keeps the top 300 most promising rectangles.
    """
    _, h, w = feature_map.shape
    total_anchors = h * w * num_anchors
    
    # Generate objectness scores (probability that each anchor contains an object)
    objectness_scores = np.random.beta(2, 10, total_anchors)  # Most are low (background)
    
    # Keep top-k proposals
    top_indices = np.argsort(objectness_scores)[-top_k:]
    
    # Generate random bounding box proposals (in real life, these come from anchor offsets)
    proposals = []
    for idx in top_indices:
        x = np.random.randint(0, 700)
        y = np.random.randint(0, 700)
        w = np.random.randint(30, 200)
        h = np.random.randint(30, 200)
        score = objectness_scores[idx]
        proposals.append([x, y, w, h, score])
    
    return np.array(proposals)


def simulate_detection_head(proposals, num_classes=3):
    """
    Simulate the detection head (classifier + box regressor).
    
    Classes: 0=background, 1=face, 2=license_plate
    """
    detections = []
    class_names = ["background", "face", "license_plate"]
    
    for proposal in proposals:
        # Simulate class probabilities (most proposals are background)
        logits = np.random.randn(num_classes)
        logits[0] += 2  # Bias toward background
        probs = np.exp(logits) / np.exp(logits).sum()
        
        predicted_class = np.argmax(probs)
        confidence = probs[predicted_class]
        
        if predicted_class > 0 and confidence > 0.3:  # Not background and confident enough
            detections.append({
                "bbox": proposal[:4].tolist(),
                "class": class_names[predicted_class],
                "confidence": float(confidence)
            })
    
    return detections


# Run the full simulated pipeline
print("=== Simulated Faster R-CNN Pipeline ===")
print()

# Stage 1: Backbone
feature_map = simulate_backbone()
print(f"1. Backbone CNN:")
print(f"   Input image: 800x800x3 (1,920,000 pixels)")
print(f"   Feature map: {feature_map.shape} ({feature_map.size:,} values)")
print(f"   Compression ratio: {800*800*3 / feature_map.size:.1f}x")

# Stage 2: RPN
proposals = simulate_rpn(feature_map)
print(f"\n2. Region Proposal Network:")
print(f"   Total anchors checked: {50*50*9:,}")
print(f"   Top proposals kept: {len(proposals)}")
print(f"   Avg objectness score: {proposals[:, 4].mean():.4f}")

# Stage 3: Detection Head
detections = simulate_detection_head(proposals)
print(f"\n3. Detection Head:")
print(f"   Proposals classified: {len(proposals)}")
print(f"   Final detections (non-background): {len(detections)}")

faces = [d for d in detections if d['class'] == 'face']
plates = [d for d in detections if d['class'] == 'license_plate']
print(f"   - Faces: {len(faces)}")
print(f"   - License plates: {len(plates)}")

## Step 6: Evaluation -- IoU, Precision, AP, and mAP

### Intersection over Union (IoU)

**12-year-old version**: If you and your friend both draw a rectangle around the same face in a photo, IoU measures how much your rectangles overlap. If they are exactly the same, IoU = 1 (perfect). If they do not overlap at all, IoU = 0 (terrible).

**Formula**: IoU = Area of Overlap / Area of Union

In [ ]:
# === IoU: The Foundation of Object Detection Evaluation ===

def compute_iou(box1, box2):
    """
    Compute Intersection over Union between two bounding boxes.
    
    Each box is [x, y, width, height] where (x, y) is the top-left corner.
    
    12-year-old version:
    Imagine two rectangles on a piece of paper.
    - Overlap area = the part where BOTH rectangles are
    - Union area = the total area covered by EITHER rectangle
    - IoU = overlap / union
    """
    x1, y1, w1, h1 = box1
    x2, y2, w2, h2 = box2
    
    # Convert to (x_min, y_min, x_max, y_max) format
    x1_min, y1_min, x1_max, y1_max = x1, y1, x1 + w1, y1 + h1
    x2_min, y2_min, x2_max, y2_max = x2, y2, x2 + w2, y2 + h2
    
    # Compute intersection
    inter_x_min = max(x1_min, x2_min)
    inter_y_min = max(y1_min, y2_min)
    inter_x_max = min(x1_max, x2_max)
    inter_y_max = min(y1_max, y2_max)
    
    inter_width = max(0, inter_x_max - inter_x_min)
    inter_height = max(0, inter_y_max - inter_y_min)
    inter_area = inter_width * inter_height
    
    # Compute union
    area1 = w1 * h1
    area2 = w2 * h2
    union_area = area1 + area2 - inter_area
    
    if union_area == 0:
        return 0.0
    
    return inter_area / union_area


# === Visualize IoU with different overlap scenarios ===

scenarios = [
    {"name": "Perfect Match", "gt": [50, 50, 100, 80], "pred": [50, 50, 100, 80]},
    {"name": "Good Overlap", "gt": [50, 50, 100, 80], "pred": [65, 55, 100, 80]},
    {"name": "Partial Overlap", "gt": [50, 50, 100, 80], "pred": [100, 70, 100, 80]},
    {"name": "No Overlap", "gt": [50, 50, 100, 80], "pred": [250, 50, 100, 80]},
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, scenario in zip(axes, scenarios):
    gt = scenario["gt"]
    pred = scenario["pred"]
    iou = compute_iou(gt, pred)
    
    ax.set_xlim(0, 400)
    ax.set_ylim(0, 200)
    ax.invert_yaxis()
    ax.set_aspect('equal')
    
    # Draw ground truth (green)
    gt_rect = patches.Rectangle((gt[0], gt[1]), gt[2], gt[3],
                                 linewidth=2, edgecolor='green', facecolor='green', alpha=0.3)
    ax.add_patch(gt_rect)
    
    # Draw prediction (red)
    pred_rect = patches.Rectangle((pred[0], pred[1]), pred[2], pred[3],
                                   linewidth=2, edgecolor='red', facecolor='red', alpha=0.3)
    ax.add_patch(pred_rect)
    
    ax.set_title(f"{scenario['name']}\nIoU = {iou:.2f}", fontsize=11, fontweight='bold')
    ax.legend(['Ground Truth', 'Prediction'], loc='lower right', fontsize=8)

plt.suptitle("Intersection over Union (IoU) Scenarios", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nIoU Thresholds and What They Mean:")
print("  IoU >= 0.5  --> 'Loose' match (PASCAL VOC standard)")
print("  IoU >= 0.75 --> 'Strict' match")
print("  IoU >= 0.5:0.95 (COCO standard) --> average across thresholds 0.5, 0.55, ..., 0.95")

In [ ]:
# === Computing Average Precision (AP) and Mean Average Precision (mAP) ===

def compute_precision_recall(detections, ground_truths, iou_threshold=0.5):
    """
    Compute precision and recall at each detection (sorted by confidence).
    
    12-year-old version:
    We sort the model's guesses from most confident to least confident.
    Then we go through them one by one and ask:
    - 'Was this guess correct?' (IoU >= threshold with a ground truth box)
    - At each step, we calculate:
      - Precision: 'Of all guesses so far, how many were right?'
      - Recall: 'Of all real objects, how many have we found so far?'
    """
    # Sort detections by confidence (highest first)
    detections = sorted(detections, key=lambda d: d['confidence'], reverse=True)
    
    num_gt = len(ground_truths)
    matched_gt = set()  # Track which ground truths have been matched
    
    precisions = []
    recalls = []
    tp_cumulative = 0
    fp_cumulative = 0
    
    for i, det in enumerate(detections):
        # Find best matching ground truth
        best_iou = 0
        best_gt_idx = -1
        
        for gt_idx, gt in enumerate(ground_truths):
            if gt_idx in matched_gt:
                continue
            iou = compute_iou(det['bbox'], gt['bbox'])
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = gt_idx
        
        if best_iou >= iou_threshold and best_gt_idx >= 0:
            tp_cumulative += 1
            matched_gt.add(best_gt_idx)
        else:
            fp_cumulative += 1
        
        precision = tp_cumulative / (tp_cumulative + fp_cumulative)
        recall = tp_cumulative / num_gt if num_gt > 0 else 0
        
        precisions.append(precision)
        recalls.append(recall)
    
    return precisions, recalls


def compute_ap(precisions, recalls):
    """
    Compute Average Precision using the 11-point interpolation method.
    
    AP = average of max precision at 11 recall levels (0.0, 0.1, ..., 1.0)
    """
    ap = 0.0
    for recall_threshold in np.arange(0, 1.1, 0.1):
        # Find max precision at recall >= threshold
        max_precision = 0
        for p, r in zip(precisions, recalls):
            if r >= recall_threshold:
                max_precision = max(max_precision, p)
        ap += max_precision
    
    return ap / 11.0


# === Example: Evaluate face detection performance ===

# Ground truth faces in an image
ground_truths_face = [
    {"bbox": [50, 50, 60, 80], "class": "face"},
    {"bbox": [200, 100, 50, 70], "class": "face"},
    {"bbox": [350, 60, 55, 75], "class": "face"},
]

# Model detections (some good, some bad)
detections_face = [
    {"bbox": [52, 48, 58, 82], "class": "face", "confidence": 0.95},  # Good match with GT[0]
    {"bbox": [198, 98, 52, 72], "class": "face", "confidence": 0.90},  # Good match with GT[1]
    {"bbox": [400, 200, 40, 50], "class": "face", "confidence": 0.85},  # False positive
    {"bbox": [345, 55, 60, 80], "class": "face", "confidence": 0.70},  # Good match with GT[2]
    {"bbox": [55, 55, 55, 75], "class": "face", "confidence": 0.60},   # Duplicate of GT[0]
    {"bbox": [150, 250, 30, 30], "class": "face", "confidence": 0.40},  # False positive
]

precisions, recalls = compute_precision_recall(detections_face, ground_truths_face, iou_threshold=0.5)
ap = compute_ap(precisions, recalls)

print("=== Face Detection Evaluation ===")
print(f"\nGround truth faces: {len(ground_truths_face)}")
print(f"Model detections: {len(detections_face)}")
print(f"\nStep-by-step (sorted by confidence):")
print(f"{'#':>3} {'Confidence':>10} {'Match?':>8} {'Precision':>10} {'Recall':>8}")
print("-" * 45)

sorted_dets = sorted(detections_face, key=lambda d: d['confidence'], reverse=True)
for i, (det, p, r) in enumerate(zip(sorted_dets, precisions, recalls)):
    is_match = p > (0 if i == 0 else precisions[i-1]) or (i == 0 and p > 0)
    match_str = "TP" if (i == 0 and p == 1.0) or (p >= precisions[i-1] if i > 0 else True) and r > (recalls[i-1] if i > 0 else 0) else "FP"
    print(f"{i+1:>3} {det['confidence']:>10.2f} {match_str:>8} {p:>10.4f} {r:>8.4f}")

print(f"\nAverage Precision (AP) for faces: {ap:.4f}")

In [ ]:
# === Precision-Recall Curve and mAP Computation ===

# Compute AP for license plates too
ground_truths_plate = [
    {"bbox": [100, 200, 80, 30], "class": "plate"},
    {"bbox": [300, 180, 75, 25], "class": "plate"},
]

detections_plate = [
    {"bbox": [102, 198, 78, 32], "class": "plate", "confidence": 0.93},
    {"bbox": [298, 178, 77, 27], "class": "plate", "confidence": 0.87},
    {"bbox": [450, 150, 60, 20], "class": "plate", "confidence": 0.55},
]

prec_plate, rec_plate = compute_precision_recall(detections_plate, ground_truths_plate)
ap_plate = compute_ap(prec_plate, rec_plate)

# Plot PR curves
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Face PR curve
axes[0].plot(recalls, precisions, 'ro-', linewidth=2, markersize=6)
axes[0].fill_between(recalls, precisions, alpha=0.2, color='red')
axes[0].set_xlabel('Recall', fontsize=12)
axes[0].set_ylabel('Precision', fontsize=12)
axes[0].set_title(f'Face Detection PR Curve\nAP = {ap:.4f}', fontsize=13, fontweight='bold')
axes[0].set_xlim([0, 1.05])
axes[0].set_ylim([0, 1.05])
axes[0].grid(True, alpha=0.3)

# Plate PR curve
axes[1].plot(rec_plate, prec_plate, 'bo-', linewidth=2, markersize=6)
axes[1].fill_between(rec_plate, prec_plate, alpha=0.2, color='blue')
axes[1].set_xlabel('Recall', fontsize=12)
axes[1].set_ylabel('Precision', fontsize=12)
axes[1].set_title(f'License Plate PR Curve\nAP = {ap_plate:.4f}', fontsize=13, fontweight='bold')
axes[1].set_xlim([0, 1.05])
axes[1].set_ylim([0, 1.05])
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Compute mAP
mAP = (ap + ap_plate) / 2
print(f"\n=== Mean Average Precision (mAP) ===")
print(f"  AP (face):          {ap:.4f}")
print(f"  AP (license plate): {ap_plate:.4f}")
print(f"  mAP:                {mAP:.4f}")
print(f"\nmAP is the GOLD STANDARD metric for object detection.")
print(f"It averages AP across ALL classes, giving a single number")
print(f"that captures overall detection quality.")

## Step 7: Serving -- The Production Pipeline

### System Architecture

```
                GOOGLE STREET VIEW BLURRING SYSTEM
    ===========================================================
    
    PIPELINE 1: BATCH PREDICTION
    
    Raw Street View Images
            |
            v
    +------------------+
    | PREPROCESSING    |  CPU-bound
    | (resize,         |  - Resize to model input size
    |  normalize,      |  - Normalize pixel values
    |  scale)          |  - Convert color space
    +------------------+
            |
            v
    +------------------+
    | DETECTION MODEL  |  GPU-bound
    | (Faster R-CNN)   |  - Run inference
    | + NMS            |  - Remove overlapping boxes
    +------------------+
            |
            v
    +------------------+
    | BLURRING         |  Apply Gaussian blur to
    | (apply blur to   |  each detected region
    |  detected areas) |
    +------------------+
            |
            v
    Object Storage (serve to users)
    
    
    PIPELINE 2: FEEDBACK LOOP
    
    User Reports (missed blurs)
            |
            v
    Hard Negative Mining --> New Training Data --> Retrain Model
```

### Key Design Decisions

1. **Separate CPU and GPU services**: Preprocessing is CPU-bound, detection is GPU-bound. Separating them allows independent scaling.
2. **Batch processing**: No real-time constraint. Process images in large batches for efficiency.
3. **Hard negative mining**: Use user reports to find and fix failure modes.

In [ ]:
# === Simulate the full serving pipeline ===

import time
from scipy.ndimage import gaussian_filter


def preprocess_image(image, target_size=(800, 800)):
    """
    Preprocess: resize and normalize.
    In production, this runs on CPUs.
    """
    # Simulate resize (in reality would use cv2.resize or PIL)
    h, w = image.shape[:2]
    # Normalize to [0, 1]
    normalized = image.astype(np.float32) / 255.0
    return normalized


def detect_objects(image):
    """
    Run object detection model.
    In production, this runs on GPUs with Faster R-CNN.
    Here we simulate with predetermined detections.
    """
    # Simulate model inference
    detections = [
        {"bbox": [70, 90, 50, 60], "class": "face", "confidence": 0.95},
        {"bbox": [240, 110, 45, 55], "class": "face", "confidence": 0.88},
        {"bbox": [310, 172, 55, 28], "class": "license_plate", "confidence": 0.92},
        {"bbox": [75, 95, 48, 58], "class": "face", "confidence": 0.72},  # overlapping with first
    ]
    return detections


def non_maximum_suppression(detections, iou_threshold=0.5):
    """
    Remove overlapping duplicate detections.
    
    12-year-old version:
    The model often draws multiple rectangles around the same face.
    NMS picks the BEST one and throws away the duplicates.
    
    Algorithm:
    1. Sort by confidence (highest first)
    2. Take the top detection -- keep it
    3. Remove any remaining detection that overlaps too much (IoU > threshold)
    4. Repeat with the next highest remaining detection
    """
    if not detections:
        return []
    
    # Sort by confidence
    sorted_dets = sorted(detections, key=lambda d: d['confidence'], reverse=True)
    
    kept = []
    while sorted_dets:
        # Keep the highest confidence detection
        best = sorted_dets.pop(0)
        kept.append(best)
        
        # Remove overlapping detections
        remaining = []
        for det in sorted_dets:
            iou = compute_iou(best['bbox'], det['bbox'])
            if iou < iou_threshold:
                remaining.append(det)  # Keep -- different object
            # else: discard -- overlapping duplicate
        sorted_dets = remaining
    
    return kept


def apply_blur(image, detections, blur_radius=10):
    """
    Apply Gaussian blur to detected regions.
    """
    blurred = image.copy()
    for det in detections:
        x, y, w, h = [int(v) for v in det['bbox']]
        # Clip to image bounds
        y_end = min(y + h, image.shape[0])
        x_end = min(x + w, image.shape[1])
        y = max(0, y)
        x = max(0, x)
        
        # Apply Gaussian blur to the region
        for c in range(3):
            blurred[y:y_end, x:x_end, c] = gaussian_filter(
                blurred[y:y_end, x:x_end, c], sigma=blur_radius
            )
    return blurred


# === Run the full pipeline ===
print("=== Full Blurring Pipeline ===")

# Create synthetic image
img = np.zeros((300, 500, 3), dtype=np.uint8)
img[:150, :] = [135, 206, 235]  # sky
img[150:, :] = [128, 128, 128]  # ground
img[50:250, 50:150] = [180, 160, 140]  # building
img[50:250, 200:350] = [160, 140, 130]  # building
img[100:140, 80:110] = [220, 180, 150]  # face
img[120:155, 250:275] = [200, 165, 135]  # face
img[180:195, 320:360] = [200, 200, 200]  # plate

# Step 1: Preprocess
preprocessed = preprocess_image(img)
print(f"1. Preprocessed image: {preprocessed.shape}, range [{preprocessed.min():.2f}, {preprocessed.max():.2f}]")

# Step 2: Detect
raw_detections = detect_objects(preprocessed)
print(f"2. Raw detections: {len(raw_detections)}")

# Step 3: NMS
final_detections = non_maximum_suppression(raw_detections)
print(f"3. After NMS: {len(final_detections)} (removed {len(raw_detections) - len(final_detections)} overlapping)")

# Step 4: Blur
blurred_img = apply_blur(img, final_detections, blur_radius=8)
print(f"4. Applied blur to {len(final_detections)} regions")

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].imshow(img)
axes[0].set_title("1. Original Image", fontsize=12, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(img)
for det in final_detections:
    x, y, w, h = det['bbox']
    color = 'red' if det['class'] == 'face' else 'blue'
    rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor=color, facecolor='none')
    axes[1].add_patch(rect)
    axes[1].text(x, y - 3, f"{det['class']} {det['confidence']:.0%}",
                 color=color, fontsize=8, fontweight='bold',
                 bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))
axes[1].set_title("2. Detections (after NMS)", fontsize=12, fontweight='bold')
axes[1].axis('off')

axes[2].imshow(blurred_img)
axes[2].set_title("3. Final Blurred Image", fontsize=12, fontweight='bold')
axes[2].axis('off')

plt.suptitle("Complete Blurring Pipeline: Original -> Detect -> Blur", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Summary: The Complete System at a Glance

| Step | Decision | Rationale |
|------|----------|----------|
| **Architecture** | Two-stage (Faster R-CNN) | Privacy demands max accuracy; offline = no speed constraint |
| **Augmentation** | Offline (1M to 10M) | Storage is cheap; faster training iteration |
| **Metric** | mAP (offline), user reports (online) | mAP captures precision-recall tradeoff; user reports measure real privacy |
| **Processing** | Batch (offline) | Requirements explicitly state no latency constraint |
| **Services** | Separate CPU/GPU | Independent scaling, better resource utilization |
| **Recall vs Precision** | Optimize for recall | Missing a face = privacy violation; false blur = cosmetic issue |
| **Post-processing** | NMS | Removes duplicate overlapping detections |
| **Feedback** | Hard negative mining | User reports create new training data for failure modes |

In [ ]:
# === Final Summary: Key Trade-offs to Remember ===

tradeoffs = [
    {
        "decision": "Two-stage vs One-stage",
        "chose": "Two-stage (Faster R-CNN)",
        "why": "Privacy demands accuracy; offline processing means speed is not critical",
        "when_to_switch": "If dataset grows 100x or we need real-time, switch to YOLO/SSD"
    },
    {
        "decision": "Offline vs Online augmentation",
        "chose": "Offline",
        "why": "Storage is cheap; avoids augmentation overhead during training",
        "when_to_switch": "If storage becomes a bottleneck or we need infinite variety"
    },
    {
        "decision": "High recall vs High precision",
        "chose": "High recall (catch everything)",
        "why": "Missing a face is a privacy violation; extra blurs are harmless",
        "when_to_switch": "Never -- privacy always wins in this domain"
    },
    {
        "decision": "Combined vs Separate CPU/GPU services",
        "chose": "Separate",
        "why": "CPU preprocessing and GPU inference scale independently",
        "when_to_switch": "At small scale, combined is simpler to manage"
    },
]

print("=== Key Trade-offs for Interview ===")
for i, t in enumerate(tradeoffs, 1):
    print(f"\n{i}. {t['decision']}")
    print(f"   Chose: {t['chose']}")
    print(f"   Why: {t['why']}")
    print(f"   Switch when: {t['when_to_switch']}")

print("\n" + "="*55)
print("\nRemember the 7-step framework:")
print("  1. Clarify requirements")
print("  2. Frame as ML task")
print("  3. Data preparation")
print("  4. Model development")
print("  5. Evaluation (IoU -> AP -> mAP)")
print("  6. Serving (NMS + batch pipeline)")
print("  7. Discussion (DETR, distributed training, GDPR, bias)")

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)